# B4 jeepney RL baseline

Traffic-biased quadrilateral baselines on the physical drivable street network.
The minimum-area rule is inspired by polygonal morphology bounds discussed in
https://arxiv.org/html/2603.28385v1.


## 1. Traffic-Biased Quadrilateral Baseline Generator

Builds traffic-weighted quadrilateral routes on the physical drive graph, then stitches the shortest physical path between the four selected anchors.

In [1]:
import pandas as pd
import secrets
from pathlib import Path
from types import SimpleNamespace

from IPython.display import IFrame, display

from _bootstrap import ROOT
from utils import BaselineRouteGenerator, JeepneySystem, JeepneyRouteEnv

OUTPUT_DIR = Path(r"C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis") / "results" / "B4_baseline_routes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_ROUTES = 20
random_seed = secrets.randbits(32)
generator = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=random_seed,
)

routes = generator.generate_routes(NUM_ROUTES, route_prefix="B4", seed=random_seed)
route_system = JeepneySystem.from_routes(routes)

summary = pd.DataFrame(
    [
        {
            "route_id": route.route_id,
            "anchors": list(route.ordered_anchor_node_ids),
            "area_m2": round(route.polygon_area_m2, 0),
            "length_m": round(route.path_length_m, 0),
            "attempt": route.attempt_index,
        }
        for route in routes
    ]
)
summary


,route_id,anchors,area_m2,length_m,attempt
0,B401,"[867, 2016, 1219, 1000]",32060889.0,42487.0,1
1,B402,"[155, 2195, 509, 961]",121640636.0,106366.0,1
2,B403,"[879, 363, 34, 1090]",4035894.0,15390.0,1
3,B404,"[87, 1457, 9, 874]",3742147.0,21079.0,1
4,B405,"[1209, 2441, 286, 198]",29338978.0,56736.0,1


## 2. RL Environment and Coordinate-Invariant Geometric Embeddings

Uses only the primal drivable network to build route geometry step by step, while encoding turn structure, sinuosity, and origin-relative features without absolute node IDs.

In [2]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)
env = JeepneyRouteEnv(
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    seed=random_seed,
    max_steps=12,
)
obs, info = env.reset()
print('reset global:', obs['global'])
print('reset candidates:\n', obs['candidates'])
print('reset mask:', obs['action_mask'])

rng = np.random.default_rng(random_seed)
for step_index in range(5):
    valid_actions = np.flatnonzero(obs['action_mask'][:-1])
    action = int(rng.choice(valid_actions)) if len(valid_actions) else env.max_candidates
    obs, reward, terminated, truncated, info = env.step(action)
    print(f'step {step_index + 1}: action={action}, reward={reward:.3f}, terminated={terminated}, truncated={truncated}')
    print('state vector:', info['state_vector'])
    print('global:', obs['global'])
    print('candidates:\n', obs['candidates'])
    print('mask:', obs['action_mask'])
    if terminated or truncated:
        break


reset global: [0.  0.  1.  0.1 0.  0.  0. ]
reset candidates:
 [[-0.04  -0.999  0.008  0.     0.     1.   ]
 [-0.038  0.999  0.009  0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.     0.   ]]
reset mask: [1. 1. 0. 0. 0. 0. 1.]
step 1: action=0, reward=-0.000, terminated=False, truncated=False
state vector: [ 0.001  0.    -1.     0.1    0.001  0.     0.     0.257  0.967  0.005
 -0.967  0.     0.     0.992  0.127  0.017 -0.127  0.     0.     0.
 -1.     0.008  1.     1.     1.     0.     0.     0.     0.     0.
  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.     0.     1.     1.     1.     0.     0.     0.     1.   ]
global: [ 0.001  0.    -1.     0.1    0.001  0.     0.   ]
candidates:
 [[ 0.257  0.967  0.005 -0.967  0.     0.   ]
 [ 0.992  0.127  0.017 -0.127  0.     0.   ]
 [ 0.    -1.     0.008  1.     1.     1. 

In [3]:
route_edges = pd.DataFrame(
    [(int(u), int(v), "start_walk") for u, v in generator.drive_graph_raw.edges()],
    columns=["u", "v", "edge_type"],
)
route_manager = SimpleNamespace(
    edges=route_edges,
    _node_coords={
        int(row.base_node_id): (float(row.lat), float(row.lon))
        for row in generator.node_table.itertuples(index=False)
    },
)

route_notes = {
    route.route_id: f"Now displaying route: {index + 1}"
    for index, route in enumerate(routes)
}

html_path = route_system.export_route_toggle_html(
    route_manager,
    OUTPUT_DIR / "B4_route_explorer.html",
    title=f"B4 Jeepney Routes ({NUM_ROUTES})",
    route_notes=route_notes,
)
print(html_path)
display(IFrame(html_path.as_uri(), width=1200, height=900))


C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\results\B4_baseline_routes\B4_route_explorer.html
